# Local AI Guardian — Deployment Notebook

This notebook takes Guardian's validated model from saved weights through
PyTorch reconstruction, ONNX export, Qualcomm AI Hub deployment on a
Snapdragon target, and finally runs the **actual shipped runtime**
(`src/guardian.py`) on two example events.

**What this notebook does *not* do:** train the model. Training happens in
the from-scratch reference script (`training/train_guardian.py`); this
notebook only reconstructs the already-trained model from
`guardian_weights.json` and takes it to deployment.

**Before running:** upload `guardian_weights.json`, `data/sample_events.json`,
`models/guardian_autoencoder.onnx`, and `src/guardian.py` (or clone your
repository, which already has this layout) into this Colab session's
working directory.

**Recommended:** run this top to bottom with *Runtime → Restart and run
all* rather than re-running individual cells out of order. An earlier
version of this notebook had a bug — a Qualcomm AI Hub variable silently
overwriting the OT device map — that only ever showed up when cells were
re-run out of sequence. Section 5 below uses `hub_devices` instead of
`devices` specifically to make that class of bug impossible.

## 0. Setup

In [ ]:
!pip install -q numpy torch onnx onnxruntime qai-hub

## 1. Load the Validated Baseline

Guardian's "normal" baseline — which devices exist and what normal
communication looks like.'

In [ ]:
import json

with open("sample_events.json") as f:
    baseline = json.load(f)

DEVICES = baseline["devices"]
NORMAL_EVENTS = baseline["events"]
print(f"Loaded {len(NORMAL_EVENTS)} baseline events across {len(DEVICES)} devices.")

SOURCES = sorted({e["source"] for e in NORMAL_EVENTS})
DESTINATIONS = sorted({e["destination"] for e in NORMAL_EVENTS})
INFORMATION_TYPES = sorted({e["information"] for e in NORMAL_EVENTS})
PROTOCOLS = sorted({(e["protocol"], e["port"]) for e in NORMAL_EVENTS})
INPUT_SIZE = len(SOURCES) + len(DESTINATIONS) + len(INFORMATION_TYPES) + len(PROTOCOLS)
print(f"Feature vocabulary -> sources:{len(SOURCES)}  destinations:{len(DESTINATIONS)}  "
      f"information:{len(INFORMATION_TYPES)}  protocols:{len(PROTOCOLS)}")
print(f"Total feature size: {INPUT_SIZE}")

Loaded 21 baseline events across 6 devices.
Feature vocabulary -> sources:5  destinations:5  information:6  protocols:2
Total feature size: 18


In [ ]:
def one_hot(value, known):
    return [1.0 if value == k else 0.0 for k in known]

def event_to_features(event):
    f = []
    f += one_hot(event["source"], SOURCES)
    f += one_hot(event["destination"], DESTINATIONS)
    f += one_hot(event["information"], INFORMATION_TYPES)
    f += one_hot((event["protocol"], event["port"]), PROTOCOLS)
    return f

print("event_to_features() defined -- encodes a communication event into an "
      f"{INPUT_SIZE}-value one-hot vector using the vocabulary above.")

event_to_features() defined -- encodes a communication event into an 18-value one-hot vector using the vocabulary above.


## 2. Reconstruct the Model in PyTorch

The weights below were already trained (see `training/train_guardian.py`
and `guardian_weights.json`). This cell only rebuilds the same
architecture in PyTorch and loads those weights directly.

In [ ]:
import torch
import torch.nn as nn

with open("guardian_weights.json") as f:
    w = json.load(f)

HIDDEN_SIZE = len(w["b1"])
print(f"Loaded guardian_weights.json  ->  input size {INPUT_SIZE}, hidden size {HIDDEN_SIZE}")

class GuardianAutoencoder(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.encoder = nn.Linear(input_size, hidden_size)
        self.decoder = nn.Linear(hidden_size, input_size)

    def forward(self, x):
        h = torch.sigmoid(self.encoder(x))
        return torch.sigmoid(self.decoder(h))

model = GuardianAutoencoder(INPUT_SIZE, HIDDEN_SIZE)
with torch.no_grad():
    model.encoder.weight.copy_(torch.tensor(w["W1"], dtype=torch.float32))
    model.encoder.bias.copy_(torch.tensor(w["b1"], dtype=torch.float32))
    model.decoder.weight.copy_(torch.tensor(w["W2"], dtype=torch.float32))
    model.decoder.bias.copy_(torch.tensor(w["b2"], dtype=torch.float32))
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"PyTorch model rebuilt from the saved weights (no retraining). Total parameters: {n_params}")

Loaded guardian_weights.json  ->  input size 18, hidden size 6
PyTorch model rebuilt from the saved weights (no retraining). Total parameters: 240


## 3. Verify PyTorch Matches the Original

This is checked against **real Guardian feature vectors** — a known-normal
event and a known-anomalous one — not random noise, since realistic
one-hot vectors are the only input distribution the model will ever
actually see in production.

In [ ]:
import numpy as np

def reconstruction_error_torch(event):
    x = torch.tensor(event_to_features(event), dtype=torch.float32)
    with torch.no_grad():
        out = model(x)
    diff = (out - x).numpy()
    return float(np.mean(diff ** 2))

print(f"{'Event':45s} {'Recon. error':>14s}")
print("-" * 61)

known_normal = NORMAL_EVENTS[12]  # PLC-01 -> HIST-01, process_data
err_normal = reconstruction_error_torch(known_normal)
label = f"{known_normal['source']} -> {known_normal['destination']} ({known_normal['information']})"
print(f"{label:45s} {err_normal:14.8f}")

reverse_event = {"source": "HIST-01", "destination": "PLC-01", "information": "process_data",
                  "protocol": "OPC-UA", "port": 4840}
err_rev = reconstruction_error_torch(reverse_event)
print(f"{'HIST-01 -> PLC-01 (reversed direction)':45s} {err_rev:14.8f}")

training_errors = [reconstruction_error_torch(e) for e in NORMAL_EVENTS]
max_error = max(training_errors)
computed_threshold = max_error * 1.5
print()
print(f"Max reconstruction error across all {len(NORMAL_EVENTS)} baseline events: {max_error:.10f}")
print(f"Computed threshold (max x 1.5):                       {computed_threshold:.10f}")
print(f"Validated runtime threshold (shipped in guardian.py):  0.0010467613755015663")
print("These agree to ~7 significant figures; the tiny remaining gap is ordinary "
      "floating-point summation-order noise and does not affect any classification.")

Event                                           Recon. error
-------------------------------------------------------------
PLC-01 -> HIST-01 (process_data)                  0.00012493
HIST-01 -> PLC-01 (reversed direction)            0.14267544

Max reconstruction error across all 21 baseline events: 0.0006978414
Computed threshold (max x 1.5):                       0.0010467621
Validated runtime threshold (shipped in guardian.py):  0.0010467613755015663
These agree to ~7 significant figures; the tiny remaining gap is ordinary floating-point summation-order noise and does not affect any classification.


## 4. Export to ONNX

Two exports: a dynamic-batch version (flexible, used by the local CPU
runtime) and a fixed-batch version (`batch_size=1`, needed for on-device
compilation in Section 5). Both are re-verified against the PyTorch
reference on every baseline event, and the dynamic export is also checked
against whichever `guardian_autoencoder.onnx` is already sitting in
`models/`, so a mismatch is caught before it is ever redeployed by
accident.

In [ ]:
import warnings, logging
warnings.filterwarnings("ignore")
logging.getLogger("torch.onnx").setLevel(logging.ERROR)

dummy = torch.zeros(1, INPUT_SIZE, dtype=torch.float32)

torch.onnx.export(
    model, dummy, "guardian_autoencoder_new.onnx",
    input_names=["guardian_input"], output_names=["guardian_output"],
    dynamic_axes={"guardian_input": {0: "batch_size"}, "guardian_output": {0: "batch_size"}},
    opset_version=13, dynamo=False,
)
torch.onnx.export(
    model, dummy, "guardian_autoencoder_static_new.onnx",
    input_names=["guardian_input"], output_names=["guardian_output"],
    opset_version=13, dynamo=False,
)
print("Exported guardian_autoencoder_new.onnx        (dynamic batch size)")
print("Exported guardian_autoencoder_static_new.onnx (fixed batch size = 1, for on-device compilation)")

import onnxruntime as ort
sess = ort.InferenceSession("guardian_autoencoder_new.onnx", providers=["CPUExecutionProvider"])
in_name = sess.get_inputs()[0].name
out_name = sess.get_outputs()[0].name

max_diff = 0.0
for e in NORMAL_EVENTS:
    x = np.array(event_to_features(e), dtype=np.float32).reshape(1, -1)
    onnx_out = sess.run([out_name], {in_name: x})[0][0]
    with torch.no_grad():
        torch_out = model(torch.tensor(x)).numpy()[0]
    max_diff = max(max_diff, float(np.abs(onnx_out - torch_out).max()))

print()
print(f"Max |ONNX - PyTorch| output difference across all {len(NORMAL_EVENTS)} baseline events: {max_diff:.2e}")
print("PASS -- the ONNX export is numerically equivalent to the PyTorch reference." if max_diff < 1e-5
      else "CHECK -- difference above tolerance, investigate before deploying.")

Exported guardian_autoencoder_new.onnx        (dynamic batch size)
Exported guardian_autoencoder_static_new.onnx (fixed batch size = 1, for on-device compilation)

Max |ONNX - PyTorch| output difference across all 21 baseline events: 1.19e-07
PASS -- the ONNX export is numerically equivalent to the PyTorch reference.


In [ ]:
sess_shipped = ort.InferenceSession("guardian_autoencoder.onnx", providers=["CPUExecutionProvider"])
in_name_s = sess_shipped.get_inputs()[0].name
out_name_s = sess_shipped.get_outputs()[0].name

max_diff_shipped = 0.0
for e in NORMAL_EVENTS:
    x = np.array(event_to_features(e), dtype=np.float32).reshape(1, -1)
    out_new = sess.run([out_name], {in_name: x})[0][0]
    out_shipped = sess_shipped.run([out_name_s], {in_name_s: x})[0][0]
    max_diff_shipped = max(max_diff_shipped, float(np.abs(out_new - out_shipped).max()))

print(f"Max difference between this freshly-exported model and the model already in "
      f"models/guardian_autoencoder.onnx: {max_diff_shipped:.2e}")
print("PASS -- identical to the model currently shipped in the repository." if max_diff_shipped < 1e-5
      else "NOTE -- this export differs from the currently-shipped model; replace it deliberately, not by accident.")

Max difference between this freshly-exported model and the model already in models/guardian_autoencoder.onnx: 0.00e+00
PASS -- identical to the model currently shipped in the repository.


## 5. Deploy via Qualcomm AI Hub

**Requires a Qualcomm AI Hub API token**  — this section makes real network calls and cannot run without one, so it is **not pre-executed** in this notebook. Run it yourself once your token is configured.

In [ ]:
from google.colab import userdata
import subprocess, getpass

try:
    api_token = userdata.get("QAI_HUB_API_TOKEN")
except Exception:
    api_token = None

if not api_token:
    api_token = getpass.getpass("Enter your Qualcomm AI Hub API token: ")

result = subprocess.run(["qai-hub", "configure", "--api_token", api_token], capture_output=True, text=True)
print(result.stdout.strip() or "Configured.")
if result.returncode != 0:
    print(result.stderr)

Enter your Qualcomm AI Hub API token: ··········
qai-hub configuration saved to /root/.qai_hub/client.ini
==================== /root/.qai_hub/client.ini ====================
[api]
api_token = pmawqfn8nj0jbzithxknv0o6lbe6w0t1yjtlqjt3
api_url = https://workbench.aihub.qualcomm.com
web_url = https://workbench.aihub.qualcomm.com
verbose = True
client_mode = cli


In [ ]:
import qai_hub as hub

target_device = hub.Device("Snapdragon X Elite CRD")

# The compiler needs a fixed input shape, so we use the static export.
compile_job = hub.submit_compile_job(
    model="guardian_autoencoder_static_new.onnx",
    device=target_device,
    input_specs=dict(guardian_input=(1, INPUT_SIZE)),
)
target_model = compile_job.get_target_model()
print("Compile job submitted:", compile_job.job_id)

Uploading guardian_autoencoder_static_new.onnx: 100%|██████████| 1.54k/1.54k [00:00<00:00, 5.67kB/s]


Scheduled compile job (jpxlzwy8p) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jpxlzwy8p/

Compile job submitted: jpxlzwy8p


In [ ]:
profile_job = hub.submit_profile_job(
    model=target_model,
    device=target_device,
)
print("Profile job submitted:", profile_job.job_id)
# profile_job.wait()
# print(profile_job.download_profile())

Scheduled profile job (j5m0lj37g) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/j5m0lj37g/

Profile job submitted: j5m0lj37g


In [ ]:
sample_event = NORMAL_EVENTS[12]  # PLC-01 -> HIST-01, process_data -- known normal
sample_input = np.array(event_to_features(sample_event), dtype=np.float32).reshape(1, -1)

inference_job = hub.submit_inference_job(
    model=target_model,
    device=target_device,
    inputs=dict(guardian_input=[sample_input]),
)
print("Inference job submitted:", inference_job.job_id)
# on_device_output = inference_job.download_output_data()
# print(on_device_output)

Uploading dataset: 100%|██████████| 13.9k/13.9k [00:00<00:00, 49.6kB/s]


Scheduled inference job (jgnzwj3jg) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jgnzwj3jg/

Inference job submitted: jgnzwj3jg


## 6. Honest Parity Check — PyTorch vs. Snapdragon

**Previously recorded result** (from an earlier run of this deployment,
using a *random* Gaussian test vector rather than a real Guardian event):
PyTorch and ONNX matched exactly; the Qualcomm-compiled output differed by
roughly 0.002 out of a reconstruction error around 0.85. That test used an
input Guardian will never actually see in production — a real one-hot
feature vector is sparse and bounded in [0, 1], not Gaussian noise — so it
measured *worst-case numeric drift*, not real-world impact.

The cell below re-runs the same comparison properly: using a real
known-normal Guardian event, and checking the number that actually
matters — **does the gap ever flip a Normal/Suspicious verdict**, given
that normal events score in the ~0.0001–0.0007 range and anomalous ones
score in the ~0.06–0.14 range (two orders of magnitude apart).

In [ ]:
output_data = inference_job.download_output_data()
print("Available output keys:", list(output_data.keys()))

output_key = list(output_data.keys())[0]  # take whatever the actual name is
on_device_output = output_data[output_key][0]

pytorch_output = model(torch.tensor(sample_input)).detach().numpy()[0]
diff = np.abs(on_device_output - pytorch_output)

recon_error_device = float(np.mean((on_device_output - sample_input[0]) ** 2))
recon_error_pytorch = float(np.mean((pytorch_output - sample_input[0]) ** 2))

print(f"Max per-element output difference (Snapdragon vs PyTorch): {diff.max():.6f}")
print(f"Reconstruction error -- PyTorch:    {recon_error_pytorch:.8f}")
print(f"Reconstruction error -- Snapdragon: {recon_error_device:.8f}")
print(f"Guardian threshold:                 0.0010467613755015663")
print()
same_verdict = (recon_error_pytorch > 0.0010467613755015663) == (recon_error_device > 0.0010467613755015663)
print("Same verdict on both platforms for this event." if same_verdict
      else "DIFFERENT VERDICT -- investigate before trusting on-device inference for this event.")

Available output keys: ['output_0']
Max per-element output difference (Snapdragon vs PyTorch): 0.003557
Reconstruction error -- PyTorch:    0.00012493
Reconstruction error -- Snapdragon: 0.00012622
Guardian threshold:                 0.0010467613755015663

Same verdict on both platforms for this event.


## 7. Run the Actual Shipped Code

Everything above reconstructs and re-verifies the model. This cell is
different: it imports and runs **`guardian.py` itself**

These two printed explanations are the real output used as the
known-normal and reverse-direction examples in the README and pitch
deck.

In [ ]:
import os, shutil, glob

os.makedirs("src", exist_ok=True)
os.makedirs("models", exist_ok=True)
os.makedirs("data", exist_ok=True)

for pattern, dest in [
    ("**/guardian.py", "src/guardian.py"),
    ("**/guardian_autoencoder.onnx", "models/guardian_autoencoder.onnx"),
    ("**/sample_events.json", "data/sample_events.json"),
]:
    matches = [m for m in glob.glob(pattern, recursive=True)
               if os.path.abspath(m) != os.path.abspath(dest)]
    if matches:
        shutil.copy(matches[0], dest)
        print(f"Copied {matches[0]} -> {dest}")
    elif os.path.exists(dest):
        print(f"Already in place: {dest}")
    else:
        print(f"NOT FOUND anywhere: {pattern} -- you'll need to upload it")

print()
print("src/ now contains:", os.listdir("src"))

Copied guardian.py -> src/guardian.py
Copied guardian_autoencoder.onnx -> models/guardian_autoencoder.onnx
Copied sample_events.json -> data/sample_events.json

src/ now contains: ['guardian.py']


In [ ]:
import sys
sys.path.insert(0, ".")
from src.guardian import analyze_event

demo_events = [
    {"source": "HMI-01", "destination": "PLC-01", "information": "process_read",
     "protocol": "Modbus/TCP", "port": 502, "timestamp": 145, "bytes": 180},
    {"source": "HIST-01", "destination": "PLC-01", "information": "process_data",
     "protocol": "OPC-UA", "port": 4840, "timestamp": 150, "bytes": 820},
]

for e in demo_events:
    outcome = analyze_event(e)
    print(outcome["explanation"])
    print("=" * 70)

✓ COMMUNICATION BEHAVIOUR NORMAL

EVENT
--------------------------------------------------
Source:        HMI-01
Destination:   PLC-01
Operation:     process_read
Protocol:      Modbus/TCP
Port:          502
Timestamp:     145 seconds
Message size:  180 bytes

WHAT GUARDIAN OBSERVED
--------------------------------------------------
The observed communication matches the behaviour expected by the Guardian baseline.

WHY WAS IT FLAGGED?
--------------------------------------------------
• No contextual, temporal, or volume anomaly detected.

ADDITIONAL CHECKS
--------------------------------------------------
AI anomaly:          NO
Context anomaly:     NO
Temporal anomaly:    NO
Volume anomaly:      NO

AI ASSESSMENT
--------------------------------------------------
Reconstruction error: 0.000231
Guardian threshold:   0.001047
The AI model considers this communication behaviour normal.

COMMUNICATION CONTEXT
--------------------------------------------------
Relationship known:       